# C.7 Reward 怎麼改變輸出機率？


只在答3與答4之間選，兩個分數都0，概率各0.5。這次人為選答4，reward=1，平常參考baseline=0.5，所以比參考好0.5。希望提高這個動作的機率，但不是一次變成必選。

policy是輸出概率分佈；reward減baseline叫advantage。令代價 `-(reward-baseline)*logp(action)`，優勢為正時，降低代價會提高所選動作log機率；為負時相反。這是策略梯度的最小例子。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

logits = torch.tensor([0.0, 0.0], requires_grad=True)
action = 1
reward, baseline = 1.0, 0.5
before = logits.detach().softmax(0)
loss = -(reward - baseline) * logits.log_softmax(0)[action]
loss.backward()
with torch.no_grad():
    updated = logits - 0.1 * logits.grad
    after = updated.softmax(0)
print("梯度", logits.grad.tolist())
print("新分數", updated.tolist())
print("前機率", before.tolist())
print("後機率", [round(p, 6) for p in after.tolist()])

梯度 `[0.25,-0.25]`，步幅0.1相減後新分數約 `[-0.025,0.025]`，概率成 `[0.487503,0.512497]`。reward與baseline是固定數字，沒有對驗證器求導。`detach()`保留舊值比較，`no_grad()`裏真的算出更新後分數；不是隻算backward就聲稱更新。


In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-C-policy-update.svg")))

這裏沒有採樣或訓練語言模型，action人工固定，直接調兩個分數。正式策略要先依分佈抽動作、取得回饋，再更新產生分數的參數。baseline是降低抽樣波動的參考，不是正確答案；可驗真值提供回饋的訓練常稱RLVR，會改權重，與上節保持權重的作答計算不同。

只把reward改0，advantage=-0.5，動作1機率反而降到約0.487503。即使單次機率按期待變化，也要在新題檢查整體行為與評分漏洞；舊嚴格reward策略更新後新題全錯，弱reward則學會列舉。真正更新和訓練回饋變高，都不能取代新題能力。

<details>
<summary>補充：舊有限策略訓練、原結果與選讀重做</summary>

正式組另外訓練一個小策略，直接讀三個0至5的整數。例如數字2用六格`[0,0,1,0,0,0]`表示，只有它對應的一格是1，其餘是0，這叫one-hot；三個數各用六格，再接在一起就是18格輸入。中間層把它們混合成48個可學特徵，再算出17種動作的分數；這些中間數字叫隱藏格，線性層怎麼混合可回看[2.3](https://birdhackor.github.io/tiny-perceptron-vlm/2.3.html)。前16種動作分別答0至15，最後一種列舉全部候選，兩種評分方式見[C.6](https://birdhackor.github.io/tiny-perceptron-vlm/C.6.html)。這個小策略與前面的兩個語言模型分開。

嚴格與弱reward兩支複製同一份隨機起點，各用167道訓練題，每批真抽64個動作。REINFORCE是這裡所用的策略梯度方法：用抽到動作的log機率與回饋估計更新方向；前面的兩分數算例則把動作人工固定，只追蹤一次更新。正式組再把梯度交給AdamW決定各參數的更新；它會累積過去梯度、調整各格步幅，並把權重衰減另行處理，方法分別見[5.4](https://birdhackor.github.io/tiny-perceptron-vlm/5.4.html)與[5.5](https://birdhackor.github.io/tiny-perceptron-vlm/5.5.html)。學習率0.003、更新1,200次，各實際抽76,800個訓練動作。

回饋不反向傳播，advantage使用抽到的reward減當下參考值。這個baseline在每批之後更新為「舊值乘0.95，加上本批平均回饋乘0.05」，供下一批參考；所以它會慢慢跟著最近回饋移動。另有0.005熵項：熵衡量候選機率分散程度，同樣17種候選全部均分時最大，全部集中一項時為0。把熵乘0.005後從訓練代價扣除，是為了鼓勵策略不要過早只押一種動作，不保證因此答對。這次確實更新策略權重，並保存兩份`policy-*.pt`，不只是前面的手寫單次分數移動。

同一24道新題在訓練前後各有一次最高機率選擇，並各抽16個動作。最高機率的分母是24題；抽樣分母是384個動作，仍只有24道不同題目：

| 策略狀態 | 最高機率動作嚴格正確 | 抽樣嚴格正確 | 抽樣弱代理分 | 抽樣列舉動作 |
| --- | --- | --- | --- | --- |
| 兩支相同起點 | 1/24 | 23/384 | 62/384 | 39/384 |
| 嚴格reward更新後 | 0/24 | 0/384 | 0/384 | 0/384 |
| 弱reward更新後 | 0/24 | 0/384 | 384/384 | 384/384 |

嚴格支線最後一批訓練抽樣reward為39/64，新題仍全部失敗，沒有支持算術泛化改善。例如`(0+3)+3=?`的真值6，它最高機率選7，16次實際抽樣也都選7。弱支線則用列舉把代理分推滿，沒有產生合格答案。訓練reward變高或真正發生梯度更新，都還需要用新題檢查能力。

這個策略選一個預定動作，沒有自回歸生成token；報告`effective_tokens=0`與`sampled_actions=76,800`各有自己的單位。它示範有限動作的policy gradient與評分漏洞，沒有替兩個語言模型做RL。可見步驟、多數一致和高reward更不能單獨作為內部推理忠實的證據。

練習只把reward改成0，baseline仍0.5。先預測advantage變負、梯度改為 `[-0.25,0.25]`，更新後動作1機率下降到約0.487503，再執行核對。請用「本次比參考差」說明方向，而不是把所有非負reward都理解成應該提高機率。

**選讀：重做本附錄的固定實驗。** 先按[W.1](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.1)在專案根目錄準備Python環境，並按[T.1](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.1)安裝Git LFS。以下入口使用固定公開資料包，完整重做短答與步驟的監督式微調（SFT，也就是學習示範回答）、多候選評估，以及有限策略的兩種回饋比較：

```bash
git lfs install --local
git lfs pull --include='assets/training/gsm8k-v1.tar.gz' --exclude=''
python scripts/course_experiments/run.py --experiment reasoning --device cpu
```

CPU執行完整排程，有自己的CUDA環境才改`--device cuda`；上文正式數字來自NVIDIA L4顯示卡。結果保存於`outputs/course-experiments/course-v1/reasoning/`，可用報告中的資料切分與各項分母核對。語言模型和有限動作策略是不同產物：`policy-*.pt`不是TinyLM權重，不能交給語言模型的推論入口。入口還包含公開長紀錄的資料通路檢查；本附錄的算術能力分數只使用上文合成的留出題。

</details>
